## load MobileNetV2 arrays

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, warnings
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
import numpy as np
import tensorflow as tf
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
PRED_DIR     = f'{PROJECT_ROOT}/results/predictions'
METRICS_DIR  = f'{PROJECT_ROOT}/results/metrics'
FIGURES_DIR  = f'{PROJECT_ROOT}/results/figures'

y_true            = np.load(f'{PRED_DIR}/y_true.npy')
y_pred_mobilenet  = np.load(f'{PRED_DIR}/y_pred_mobilenet.npy')
uncertainty       = np.load(f'{PRED_DIR}/uncertainty_mobilenet.npy')

print(f"y_true:      {y_true.shape}")
print(f"y_pred:      {y_pred_mobilenet.shape}")
print(f"uncertainty: {uncertainty.shape}")
print(f"Overall acc: {np.mean(y_pred_mobilenet == y_true):.4f}  (expected 0.9231)")
print(f"\nPer-class accuracy:")
for i, cls in enumerate(CLASS_NAMES):
    mask = y_true == i
    acc  = np.mean(y_pred_mobilenet[mask] == y_true[mask])
    unc  = uncertainty[mask].mean()
    print(f"  {cls:>12}: acc={acc:.4f}  mean_unc={unc:.4f}")

print("\nDay 2 ready -- Fairness Audit + Robustness (MobileNetV2)")


## 1. Reuse simulated demographics (didnt re-simulated blindly ,verified determinism)

In [ ]:
import numpy as np

# Load the demographics already simulated during the EfficientNetB3 run
gender    = np.load(f'{PRED_DIR}/gender_labels.npy')
age_group = np.load(f'{PRED_DIR}/age_group_labels.npy')

print(f"Loaded existing demographics:")
print(f"  Male: {(gender==0).sum()}  Female: {(gender==1).sum()}")
print(f"  18-40: {(age_group==0).sum()}  41-60: {(age_group==1).sum()}  61+: {(age_group==2).sum()}")

# VERIFY determinism rather than assume it: re-run the exact same simulation
# logic against the same y_true and seed, and confirm it reproduces the
# saved arrays exactly. If this assertion ever fails, the two models were
# evaluated against different y_true orderings -- stop immediately.
np.random.seed(SEED)
n = len(y_true)
gender_check = np.zeros(n, dtype=int)
for i in range(n):
    cls = y_true[i]
    if cls == 0:    gender_check[i] = np.random.choice([0,1], p=[0.60, 0.40])
    elif cls == 1:  gender_check[i] = np.random.choice([0,1], p=[0.35, 0.65])
    elif cls == 2:  gender_check[i] = np.random.choice([0,1], p=[0.50, 0.50])
    else:           gender_check[i] = np.random.choice([0,1], p=[0.45, 0.55])

age_group_check = np.zeros(n, dtype=int)
for i in range(n):
    cls = y_true[i]
    if cls == 0:    age_group_check[i] = np.random.choice([0,1,2], p=[0.25, 0.50, 0.25])
    elif cls == 1:  age_group_check[i] = np.random.choice([0,1,2], p=[0.15, 0.40, 0.45])
    elif cls == 2:  age_group_check[i] = np.random.choice([0,1,2], p=[0.45, 0.35, 0.20])
    else:           age_group_check[i] = np.random.choice([0,1,2], p=[0.50, 0.35, 0.15])

assert np.array_equal(gender, gender_check), (
    "STOP: re-simulated gender labels do not match the saved file. "
    "This means y_true ordering differs between the EfficientNetB3 and "
    "MobileNetV2 runs -- do not proceed until this is understood, since "
    "every downstream fairness number depends on this alignment."
)
assert np.array_equal(age_group, age_group_check), (
    "STOP: re-simulated age-group labels do not match the saved file."
)
print("\nVerified: demographics are deterministic and identical across "
      "both backbones (same y_true, same seed). Safe to proceed.")

gender_labels = np.array(['Male','Female'])[gender]
age_labels    = np.array(['18-40','41-60','61+'])[age_group]


## 2. Fairlearn -- Gender & Age Group Accuracy/Uncertainty

In [ ]:
print("="*55)
print("FAIRLEARN -- Gender Analysis (MobileNetV2)")
print("="*55)

for grp in ['Male','Female']:
    mask = gender_labels == grp
    acc  = np.mean(y_pred_mobilenet[mask] == y_true[mask])
    unc  = uncertainty[mask].mean()
    print(f"  {grp:>8}: acc={acc:.4f}  mean_uncertainty={unc:.4f}  n={mask.sum()}")

acc_male   = np.mean(y_pred_mobilenet[gender_labels=='Male']   == y_true[gender_labels=='Male'])
acc_female = np.mean(y_pred_mobilenet[gender_labels=='Female'] == y_true[gender_labels=='Female'])
gender_gap = abs(acc_male - acc_female)
print(f"\n  Gender accuracy gap: {gender_gap:.4f} ({gender_gap*100:.2f}%)")
print(f"  (For reference, EfficientNetB3's gender gap was 1.77%)")

print("\n" + "="*55)
print("FAIRLEARN -- Age Group Analysis (MobileNetV2)")
print("="*55)

n = len(y_true)
age_accs, age_uncs = {}, {}
for grp in ['18-40','41-60','61+']:
    mask = age_labels == grp
    acc  = np.mean(y_pred_mobilenet[mask] == y_true[mask])
    unc  = uncertainty[mask].mean()
    age_accs[grp] = acc
    age_uncs[grp] = unc
    print(f"  {grp:>8}: acc={acc:.4f}  mean_uncertainty={unc:.4f}  n={mask.sum()}")

age_gap = max(age_accs.values()) - min(age_accs.values())
print(f"\n  Age group accuracy gap: {age_gap:.4f} ({age_gap*100:.2f}%)")
print(f"  (For reference, EfficientNetB3's 18-40 vs 61+ gap was 3.59%)")

print("\n" + "="*55)
print("NOVEL: High-uncertainty predictions by demographic group (MobileNetV2)")
print("="*55)

high_unc_threshold = np.percentile(uncertainty, 80)
high_unc_mask = uncertainty >= high_unc_threshold
print(f"\n  High uncertainty threshold (80th percentile): {high_unc_threshold:.4f}")
print(f"  High uncertainty predictions: {high_unc_mask.sum()}")

gender_uncertainty_summary = {}
print("\n  Gender distribution in high-uncertainty predictions:")
for grp in ['Male','Female']:
    total_grp    = (gender_labels==grp).sum()
    high_unc_grp = (high_unc_mask & (gender_labels==grp)).sum()
    pct = high_unc_grp / high_unc_mask.sum() * 100
    base_pct = total_grp / n * 100
    gender_uncertainty_summary[grp] = {'pct_of_uncertain': pct, 'pct_of_total': base_pct}
    print(f"    {grp:>8}: {high_unc_grp}/{high_unc_mask.sum()} "
          f"({pct:.1f}% of uncertain) vs {base_pct:.1f}% of total")

age_uncertainty_summary = {}
print("\n  Age group distribution in high-uncertainty predictions:")
for grp in ['18-40','41-60','61+']:
    total_grp    = (age_labels==grp).sum()
    high_unc_grp = (high_unc_mask & (age_labels==grp)).sum()
    pct = high_unc_grp / high_unc_mask.sum() * 100
    base_pct = total_grp / n * 100
    age_uncertainty_summary[grp] = {'pct_of_uncertain': pct, 'pct_of_total': base_pct}
    print(f"    {grp:>8}: {high_unc_grp}/{high_unc_mask.sum()} "
          f"({pct:.1f}% of uncertain) vs {base_pct:.1f}% of total")
print(f"\n  (For reference, EfficientNetB3: 61+ was 38.1% of uncertain vs 26.3% "
      f"of total -- an 11.8pp gap)")

import json
fairness_metrics_mobilenet = {
    'note': 'Demographics are shared with the EfficientNetB3 run (verified '
            'deterministic in Cell 1) -- simulated, not real patient data.',
    'gender': {'male_acc': float(acc_male), 'female_acc': float(acc_female), 'gap': float(gender_gap)},
    'age_group': {
        'acc_18_40': float(age_accs['18-40']), 'acc_41_60': float(age_accs['41-60']),
        'acc_61_plus': float(age_accs['61+']), 'gap': float(age_gap)
    },
    'high_uncertainty_threshold_percentile': 80,
    'high_uncertainty_threshold_value': float(high_unc_threshold),
    'gender_high_uncertainty_distribution': gender_uncertainty_summary,
    'age_high_uncertainty_distribution': age_uncertainty_summary
}
with open(f'{METRICS_DIR}/fairness_metrics_mobilenet.json','w') as f:
    json.dump(fairness_metrics_mobilenet, f, indent=2)
print("\nSaved fairness_metrics_mobilenet.json")


## 3. AIF360 -- Disparate Impact + Reweighing (Gender), Disparate Impact (Age)

In [ ]:
!pip install -q aif360
import numpy as np, json, pandas as pd
import matplotlib.pyplot as plt
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

print("="*55)
print("AIF360 -- Disparate Impact Analysis (Gender, MobileNetV2)")
print("="*55)

y_correct = (y_pred_mobilenet == y_true).astype(float)
df_gender = pd.DataFrame({'y_true': y_correct, 'gender': gender})

aif_dataset = BinaryLabelDataset(
    df=df_gender, label_names=['y_true'], protected_attribute_names=['gender'],
    favorable_label=1.0, unfavorable_label=0.0
)
metric_orig = BinaryLabelDatasetMetric(
    aif_dataset, unprivileged_groups=[{'gender': 1}], privileged_groups=[{'gender': 0}]
)
di_before  = metric_orig.disparate_impact()
spd_before = metric_orig.statistical_parity_difference()

acc_male_bef   = np.mean(y_pred_mobilenet[gender==0]==y_true[gender==0])
acc_female_bef = np.mean(y_pred_mobilenet[gender==1]==y_true[gender==1])
print(f"\nBEFORE Reweighing:")
print(f"  Disparate Impact Ratio:         {di_before:.4f}  (ideal=1.0, >0.8 acceptable)")
print(f"  Statistical Parity Difference:  {spd_before:.4f} (ideal=0.0)")
print(f"  Male accuracy:                  {acc_male_bef:.4f}")
print(f"  Female accuracy:                {acc_female_bef:.4f}")

RW = Reweighing(unprivileged_groups=[{'gender': 1}], privileged_groups=[{'gender': 0}])
aif_reweighed = RW.fit_transform(aif_dataset)
weights = aif_reweighed.instance_weights

acc_male_rw   = np.average(y_pred_mobilenet[gender==0]==y_true[gender==0], weights=weights[gender==0])
acc_female_rw = np.average(y_pred_mobilenet[gender==1]==y_true[gender==1], weights=weights[gender==1])
metric_rw = BinaryLabelDatasetMetric(
    aif_reweighed, unprivileged_groups=[{'gender': 1}], privileged_groups=[{'gender': 0}]
)
di_after, spd_after = metric_rw.disparate_impact(), metric_rw.statistical_parity_difference()

print(f"\nAFTER Reweighing:")
print(f"  Disparate Impact Ratio:         {di_after:.4f}")
print(f"  Statistical Parity Difference:  {spd_after:.4f}")
print(f"  Male accuracy (weighted):       {acc_male_rw:.4f}")
print(f"  Female accuracy (weighted):     {acc_female_rw:.4f}")

print("\n" + "="*55)
print("AIF360 -- Disparate Impact Analysis (Age Group, MobileNetV2)")
print("="*55)
mask_young, mask_old = age_group==0, age_group==2
acc_young = np.mean(y_pred_mobilenet[mask_young] == y_true[mask_young])
acc_old   = np.mean(y_pred_mobilenet[mask_old]   == y_true[mask_old])
di_age    = acc_old/acc_young if acc_young>0 else 0
print(f"\n  18-40 accuracy: {acc_young:.4f}")
print(f"  61+   accuracy: {acc_old:.4f}")
print(f"  Disparate Impact (61+ vs 18-40): {di_age:.4f}  (>0.8 acceptable)")
print(f"  Gap: {abs(acc_young-acc_old)*100:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
groups = ['Male', 'Female']
acc_bef = [acc_male_bef, acc_female_bef]
acc_aft = [acc_male_rw, acc_female_rw]
x = np.arange(2)
axes[0].bar(x-0.2, acc_bef, 0.35, label='Before Reweighing', color='steelblue', alpha=0.8)
axes[0].bar(x+0.2, acc_aft, 0.35, label='After Reweighing', color='coral', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(groups)
axes[0].set_ylabel('Accuracy'); axes[0].set_title('Gender -- Before vs After (MobileNetV2)', fontsize=11)
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, axis='y')

age_groups_lbl = ['18-40', '41-60', '61+']
age_accs_vals = [
    np.mean(y_pred_mobilenet[age_group==0]==y_true[age_group==0]),
    np.mean(y_pred_mobilenet[age_group==1]==y_true[age_group==1]),
    np.mean(y_pred_mobilenet[age_group==2]==y_true[age_group==2])
]
axes[1].bar(age_groups_lbl, age_accs_vals, color=['steelblue','orange','coral'], alpha=0.8)
axes[1].set_ylabel('Accuracy'); axes[1].set_title('Age Group Accuracy (MobileNetV2)', fontsize=11)
axes[1].grid(alpha=0.3, axis='y')
for i, v in enumerate(age_accs_vals):
    axes[1].text(i, v+0.001, f'{v:.3f}', ha='center', fontsize=9)

age_uncs_vals = [uncertainty[age_group==0].mean(), uncertainty[age_group==1].mean(), uncertainty[age_group==2].mean()]
axes[2].bar(age_groups_lbl, age_uncs_vals, color=['steelblue','orange','coral'], alpha=0.8)
axes[2].set_ylabel('Mean Uncertainty'); axes[2].set_title('Mean Uncertainty by Age Group (MobileNetV2)', fontsize=11)
axes[2].grid(alpha=0.3, axis='y')
for i, v in enumerate(age_uncs_vals):
    axes[2].text(i, v+0.0003, f'{v:.4f}', ha='center', fontsize=9)

plt.suptitle('Fairness Audit -- MobileNetV2 Brain Tumor Classifier', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fairness_audit_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()

fairness_report_mobilenet = {
    'disclaimer': 'Demographic labels are simulated (shared with the EfficientNetB3 '
                  'run, verified identical in Cell 1); results demonstrate the audit '
                  'methodology, not real-world disparities.',
    'gender_audit': {
        'disparate_impact_before': float(di_before), 'disparate_impact_after': float(di_after),
        'spd_before': float(spd_before), 'spd_after': float(spd_after),
        'male_acc': float(acc_male_bef), 'female_acc': float(acc_female_bef),
        'gap': float(abs(acc_male_bef-acc_female_bef)), 'mitigation': 'Reweighing (AIF360)'
    },
    'age_audit': {
        'acc_18_40': float(age_accs_vals[0]), 'acc_41_60': float(age_accs_vals[1]),
        'acc_61_plus': float(age_accs_vals[2]), 'disparate_impact_61_vs_1840': float(di_age),
        'gap': float(abs(age_accs_vals[0]-age_accs_vals[2]))
    }
}
with open(f'{METRICS_DIR}/fairness_report_complete_mobilenet.json', 'w') as f:
    json.dump(fairness_report_mobilenet, f, indent=2)
print("\nSaved fairness_report_complete_mobilenet.json")


## 4. Fairlearn -- Equalized Odds + Demographic Parity (per class, OvR)

In [ ]:
!pip install -q fairlearn
import numpy as np, json
import matplotlib.pyplot as plt
from fairlearn.metrics import equalized_odds_difference, demographic_parity_difference

print("="*55)
print("FAIRLEARN -- Equalized Odds (per class, OvR, MobileNetV2)")
print("="*55)

eod_results, dpd_results = {}, {}
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    y_bin      = (y_true == cls_idx).astype(int)
    y_pred_bin = (y_pred_mobilenet == cls_idx).astype(int)

    eod_gender = equalized_odds_difference(y_bin, y_pred_bin, sensitive_features=gender_labels)
    dpd_gender = demographic_parity_difference(y_bin, y_pred_bin, sensitive_features=gender_labels)
    eod_age    = equalized_odds_difference(y_bin, y_pred_bin, sensitive_features=age_labels)
    dpd_age    = demographic_parity_difference(y_bin, y_pred_bin, sensitive_features=age_labels)

    eod_results[cls_name] = {'gender': float(eod_gender), 'age': float(eod_age)}
    dpd_results[cls_name] = {'gender': float(dpd_gender), 'age': float(dpd_age)}

    print(f"\n  {cls_name.upper()}")
    print(f"    Equalized Odds Diff (gender): {eod_gender:.4f}  (ideal=0, <0.1 good)")
    print(f"    Equalized Odds Diff (age):    {eod_age:.4f}")
    print(f"    Demographic Parity Diff (gender): {dpd_gender:.4f}")
    print(f"    Demographic Parity Diff (age):    {dpd_age:.4f}")
    if eod_gender >= 0.1 or eod_age >= 0.1:
        print(f"    *** NOTE: exceeds the 0.1 Equalized-Odds threshold -- flag in writeup ***")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(CLASS_NAMES))
eod_g = [eod_results[c]['gender'] for c in CLASS_NAMES]
eod_a = [eod_results[c]['age']    for c in CLASS_NAMES]
dpd_g = [dpd_results[c]['gender'] for c in CLASS_NAMES]
dpd_a = [dpd_results[c]['age']    for c in CLASS_NAMES]

axes[0].bar(x-0.2, eod_g, 0.35, label='Gender', color='steelblue', alpha=0.8)
axes[0].bar(x+0.2, eod_a, 0.35, label='Age', color='coral', alpha=0.8)
axes[0].axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='Threshold (0.1)')
axes[0].set_xticks(x); axes[0].set_xticklabels([c.capitalize() for c in CLASS_NAMES])
axes[0].set_ylabel('Equalized Odds Difference'); axes[0].set_title('Equalized Odds -- MobileNetV2', fontsize=11)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(x-0.2, dpd_g, 0.35, label='Gender', color='steelblue', alpha=0.8)
axes[1].bar(x+0.2, dpd_a, 0.35, label='Age', color='coral', alpha=0.8)
axes[1].axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='Threshold (0.05)')
axes[1].set_xticks(x); axes[1].set_xticklabels([c.capitalize() for c in CLASS_NAMES])
axes[1].set_ylabel('Demographic Parity Difference'); axes[1].set_title('Demographic Parity -- MobileNetV2', fontsize=11)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Fairlearn -- Equalized Odds & Demographic Parity (MobileNetV2)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/equalized_odds_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()

fairness_eq_mobilenet = {'equalized_odds_difference': eod_results, 'demographic_parity_difference': dpd_results}
with open(f'{METRICS_DIR}/equalized_odds_mobilenet.json', 'w') as f:
    json.dump(fairness_eq_mobilenet, f, indent=2)
print("\nSaved equalized_odds_mobilenet.json")


## 5. Artifact Robustness -- Gaussian Blur, Brightness, JPEG

In [ ]:
import numpy as np, matplotlib.pyplot as plt, cv2, os, json
import tensorflow as tf

IMG_SIZE, BATCH_SIZE = (224, 224), 32
CLAHE_DIR = f'{PROJECT_ROOT}/data/clahe_processed'

mobilenet_model = tf.keras.models.load_model(f'{PROJECT_ROOT}/models/checkpoints/mobilenetv2_clean.keras')
preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input

print("Loading test images for robustness testing...")
test_ds = tf.keras.utils.image_dataset_from_directory(
    f'{CLAHE_DIR}/Testing', seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', class_names=CLASS_NAMES, shuffle=False
)
X_list, y_list = [], []
for x_batch, y_batch in test_ds:
    X_list.append(x_batch.numpy()); y_list.append(y_batch.numpy())
X_test  = np.concatenate(X_list, axis=0).astype(np.uint8)
y_check = np.concatenate(y_list, axis=0)
del X_list, y_list
print(f"Loaded: {X_test.shape}, labels match: {np.array_equal(y_check, y_true)}")

def apply_gaussian_blur(images, k):
    return np.array([cv2.GaussianBlur(img, (k, k), 0) for img in images])
def apply_brightness_shift(images, delta):
    return np.clip(images.astype(np.int16) + delta, 0, 255).astype(np.uint8)
def apply_jpeg_compression(images, quality):
    result = []
    for img in images:
        _, enc = cv2.imencode('.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        result.append(cv2.cvtColor(dec, cv2.COLOR_BGR2RGB))
    return np.array(result)

def predict_accuracy(images):
    preprocessed = preprocess_fn(images.astype(np.float32))
    preds = []
    for i in range(0, len(preprocessed), BATCH_SIZE):
        batch = tf.constant(preprocessed[i:i+BATCH_SIZE])
        pred  = mobilenet_model(batch, training=False).numpy()
        preds.append(np.argmax(pred, axis=1))
    return float(np.mean(np.concatenate(preds) == y_true))

print("\nRunning robustness tests (MobileNetV2)...")
baseline_acc = float(np.mean(y_pred_mobilenet == y_true))
results = {'baseline': baseline_acc}
print(f"  baseline (sanity check): {baseline_acc:.4f} (expected 0.9231)")

print("  Gaussian blur...")
for k in [3, 7, 15]:
    acc = predict_accuracy(apply_gaussian_blur(X_test, k))
    results[f'gaussian_blur_k{k}'] = acc
    print(f"    kernel={k}: {acc:.4f}")

print("  Brightness shift...")
for delta in [-50, 50, 80]:
    acc = predict_accuracy(apply_brightness_shift(X_test, delta))
    results[f'brightness_{"+"+str(delta) if delta>0 else str(delta)}'] = acc
    print(f"    delta={delta:+d}: {acc:.4f}")

print("  JPEG compression...")
for q in [80, 50, 20]:
    acc = predict_accuracy(apply_jpeg_compression(X_test, q))
    results[f'jpeg_q{q}'] = acc
    print(f"    quality={q}: {acc:.4f}")

print("\nAll robustness tests complete.")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
blur_ks = [3, 7, 15]
blur_accs = [results[f'gaussian_blur_k{k}'] for k in blur_ks]
axes[0].plot(blur_ks, blur_accs, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axhline(y=baseline_acc, color='gray', linestyle='--', label=f'Baseline ({baseline_acc:.3f})')
axes[0].set_xlabel('Gaussian Kernel Size'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Robustness -- Gaussian Blur (MobileNetV2)', fontsize=11)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3); axes[0].set_ylim(0.0, 1.0)
for k, acc in zip(blur_ks, blur_accs):
    axes[0].annotate(f'{acc:.3f}', (k, acc), textcoords="offset points", xytext=(0,8), ha='center', fontsize=9)

br_deltas = [-50, 50, 80]
br_accs = [results[f'brightness_{"+"+str(d) if d>0 else str(d)}'] for d in br_deltas]
axes[1].bar([str(d) for d in br_deltas], br_accs, color=['coral','steelblue','orange'], alpha=0.8)
axes[1].axhline(y=baseline_acc, color='gray', linestyle='--', label=f'Baseline ({baseline_acc:.3f})')
axes[1].set_xlabel('Brightness Delta'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Robustness -- Brightness Shift (MobileNetV2)', fontsize=11)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, axis='y'); axes[1].set_ylim(0.0, 1.0)
for i, acc in enumerate(br_accs):
    axes[1].text(i, acc+0.005, f'{acc:.3f}', ha='center', fontsize=9)

jpeg_qs = [80, 50, 20]
jpeg_accs = [results[f'jpeg_q{q}'] for q in jpeg_qs]
axes[2].plot(jpeg_qs, jpeg_accs, 's-', color='coral', linewidth=2, markersize=8)
axes[2].axhline(y=baseline_acc, color='gray', linestyle='--', label=f'Baseline ({baseline_acc:.3f})')
axes[2].set_xlabel('JPEG Quality'); axes[2].set_ylabel('Accuracy')
axes[2].set_title('Robustness -- JPEG Compression (MobileNetV2)', fontsize=11)
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3); axes[2].set_ylim(0.0, 1.0); axes[2].invert_xaxis()
for q, acc in zip(jpeg_qs, jpeg_accs):
    axes[2].annotate(f'{acc:.3f}', (q, acc), textcoords="offset points", xytext=(0,8), ha='center', fontsize=9)

plt.suptitle('MRI Artifact Robustness -- MobileNetV2', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/artifact_robustness_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()

with open(f'{METRICS_DIR}/artifact_robustness_mobilenet.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved artifact_robustness_mobilenet.json")


## 6. Motion Blur + Additive Gaussian Sensor Noise

In [ ]:
import numpy as np, matplotlib.pyplot as plt, cv2, json

def apply_motion_blur(images, kernel_size, angle_deg=0):
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[kernel_size // 2, :] = 1.0
    center = (kernel_size / 2 - 0.5, kernel_size / 2 - 0.5)
    rot_mat = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    kernel = cv2.warpAffine(kernel, rot_mat, (kernel_size, kernel_size))
    kernel = kernel / kernel.sum()
    return np.array([cv2.filter2D(img, -1, kernel) for img in images])

def apply_gaussian_noise(images, sigma):
    noisy = images.astype(np.float32) + np.random.normal(0, sigma, images.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)

results2 = {'baseline': baseline_acc}

print("Running motion-blur sweep (MobileNetV2)...")
motion_lengths = [5, 9, 15]
for length in motion_lengths:
    acc = predict_accuracy(apply_motion_blur(X_test, length, angle_deg=30))
    results2[f'motion_blur_len{length}'] = acc
    print(f"  length={length}px: {acc:.4f}")

print("\nRunning Gaussian noise sweep (MobileNetV2)...")
noise_sigmas = [5, 15, 25]
for sigma in noise_sigmas:
    acc = predict_accuracy(apply_gaussian_noise(X_test, sigma))
    results2[f'gaussian_noise_sigma{sigma}'] = acc
    print(f"  sigma={sigma}: {acc:.4f}")

print("\nAll motion/noise tests complete.")
print(f"  (For reference, EfficientNetB3: motion blur 15px -> 57.13%, noise sigma=25 -> 73.25%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
mb_accs = [results2[f'motion_blur_len{l}'] for l in motion_lengths]
axes[0].plot(motion_lengths, mb_accs, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[0].axhline(y=baseline_acc, color='gray', linestyle='--', label=f'Baseline ({baseline_acc:.3f})')
axes[0].set_xlabel('Motion Blur Length (px)'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Robustness -- Motion Blur (MobileNetV2)', fontsize=11)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3); axes[0].set_ylim(0.0, 1.0)
for l, acc in zip(motion_lengths, mb_accs):
    axes[0].annotate(f'{acc:.3f}', (l, acc), textcoords="offset points", xytext=(0,8), ha='center', fontsize=9)

gn_accs = [results2[f'gaussian_noise_sigma{s}'] for s in noise_sigmas]
axes[1].plot(noise_sigmas, gn_accs, 's-', color='coral', linewidth=2, markersize=8)
axes[1].axhline(y=baseline_acc, color='gray', linestyle='--', label=f'Baseline ({baseline_acc:.3f})')
axes[1].set_xlabel('Gaussian Noise sigma'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Robustness -- Additive Gaussian Noise (MobileNetV2)', fontsize=11)
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3); axes[1].set_ylim(0.0, 1.0)
for s, acc in zip(noise_sigmas, gn_accs):
    axes[1].annotate(f'{acc:.3f}', (s, acc), textcoords="offset points", xytext=(0,8), ha='center', fontsize=9)

plt.suptitle('MRI Robustness -- Motion Blur & Sensor Noise (MobileNetV2)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/motion_noise_robustness_mobilenet.png', dpi=150, bbox_inches='tight')
plt.show()

with open(f'{METRICS_DIR}/motion_noise_robustness_mobilenet.json', 'w') as f:
    json.dump(results2, f, indent=2)
print("\nSaved motion_noise_robustness_mobilenet.json")
